# Machine Translation

# Dataset Loading and Splitting

This code block handles dataset loading and splitting for training the machine translation model.

- Standard libraries like `torch`, `nn`, and `DataLoader` are imported for PyTorch-based model training.
- The Dutch (`Source.nl`) and English (`translations_scored.en`) files are read line by line into Python lists.
- `LIMIT = 100000` restricts the dataset size to the first 100,000 lines, which helps reduce memory usage and training time during experiments.
- `assert` ensures the source and target files are aligned and contain the same number of sentences.
- The data is split into training (90%) and validation (10%) sets:

```python
split_idx = int(0.9 * len(src_texts))
train_src, valid_src = src_texts[:split_idx], src_texts[split_idx:]
train_tgt, valid_tgt = tgt_texts[:split_idx], tgt_texts[split_idx:]
```
This ensures the model has data to learn from (train_) and separate data to evaluate generalization (valid_).

In [12]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from torch import nn
import random
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

with open("./Translations/Source.nl", "r", encoding="utf-8") as f:
    src_texts = [line.strip() for line in f.readlines()]

with open("./Translations/translations_scored.en", "r", encoding="utf-8") as f:
    tgt_texts = [line.strip() for line in f.readlines()]

LIMIT = 100000
src_texts = src_texts[:LIMIT]
tgt_texts = tgt_texts[:LIMIT]

assert len(src_texts) == len(tgt_texts), "Source and target files must have the same number of lines"

split_idx = int(0.9 * len(src_texts))
train_src, valid_src = src_texts[:split_idx], src_texts[split_idx:]
train_tgt, valid_tgt = tgt_texts[:split_idx], tgt_texts[split_idx:]

# Tokenizer Training with BPE

This code block sets up and trains a Byte-Pair Encoding (BPE) tokenizer for both Dutch and English sentences.

- `Tokenizer(models.BPE())` initializes a BPE tokenizer from scratch.
- `tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()` ensures that input text is first split on whitespace before applying BPE merges.
- `BpeTrainer` specifies:
  - `vocab_size=100000` to create a large vocabulary that reduces the number of unknown tokens, this is however greater than the dataset at this moment so not really needed anymore (still adjustable to help with training times).
  - `min_frequency=2` to ignore very rare tokens, keeping the vocabulary efficient (also very adjustable, still not sure what the best option is here for now 2 will do).
  - `special_tokens=["<unk>", "<pad>", "<bos>", "<eos>"]` which are required for sequence modeling:
    - `<unk>`: unknown token for rare words  
    - `<pad>`: padding for uniform sequence lengths  
    - `<bos>`: beginning-of-sentence token  
    - `<eos>`: end-of-sentence token

- `get_training_corpus()` yields all sentences from the training set, interleaving Dutch and English so the tokenizer learns subwords for both languages.
- `tokenizer.train_from_iterator()` trains the tokenizer on this corpus.
- `print(tokenizer.get_vocab_size())` confirms the vocabulary size after training.

This setup allows the model to handle subwords efficiently and reduces `<unk>` tokens during translation. However, we can not be totally sure that the translations are 100% as of yet.

In [13]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=100000,
    min_frequency=2,
    special_tokens=["<unk>", "<pad>", "<bos>", "<eos>"]
)

def get_training_corpus():
    for src, tgt in zip(train_src, train_tgt):
        yield src
        yield tgt

# Train the tokenizer
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
print("Tokenizer trained with vocab size:", tokenizer.get_vocab_size())

Tokenizer trained with vocab size: 37894


# Dataset Preparation and DataLoader Setup

Defining how sentences are encoded, batched, and fed to the model.

- `encode_sentence(sentence, max_length=32)` converts a sentence into token IDs using the BPE tokenizer:
  - `ids = tokenizer.encode(sentence).ids` encodes the sentence.
  - `ids = ids[:max_length]` truncates sentences that are too long.
  - Padding is added with the `<pad>` token to ensure uniform length across a batch:
    ```python
    padding = [tokenizer.token_to_id("<pad>")] * (max_length - len(ids))
    return ids + padding
    ```

- `TranslationDataset` is a PyTorch Dataset class:
  - Stores source and target sentences.
  - `__getitem__` returns a tensor of token IDs for a single source-target pair.
  
- `collate_fn(batch)` takes a batch of samples and:
  - Pads all sequences so they have the same length.
  - Returns src_batch and tgt_batch in shape `[seq_len, batch_size]` for the model.

- `DataLoader` creates iterators for training and validation:
  - `train_loader` shuffles the dataset to improve learning.
  - `valid_loader` does not shuffle, ensuring consistent evaluation (basically meaning the epochs are validated on the same data to ensure consitent results and minimizing the amount of variables).
  - `batch_size=32` balances GPU memory usage and training speed.

This setup ensures that each batch is properly padded, truncated, and ready for the Seq2Seq model.

In [14]:
def encode_sentence(sentence, max_length=32):
    ids = tokenizer.encode(sentence).ids
    ids = ids[:max_length]
    padding = [tokenizer.token_to_id("<pad>")] * (max_length - len(ids))
    return ids + padding

class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src = encode_sentence(self.src_texts[idx])
        tgt = encode_sentence(self.tgt_texts[idx])
        return torch.tensor(src), torch.tensor(tgt)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=tokenizer.token_to_id("<pad>"))
    tgt_batch = pad_sequence(tgt_batch, padding_value=tokenizer.token_to_id("<pad>"))
    return src_batch, tgt_batch

train_loader = DataLoader(
    TranslationDataset(train_src, train_tgt),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    TranslationDataset(valid_src, valid_tgt),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


# Seq2Seq Model: Encoder, Decoder, and Full Architecture

Defining the core Seq2Seq model for machine translation using GRU cells. I chose GRUs over LSTMs because they have fewer parameters and train faster while still effectively capturing sequence dependencies, making them well-suited for medium-sized translation datasets.

The Encoder converts input token IDs into embeddings (`nn.Embedding`) and processes them with a GRU. It returns the final hidden state, which summarizes the input sequence. The parameters are:

- `input_dim`: vocabulary size
- `emb_dim`: embedding dimension
- `hid_dim`: GRU hidden size

The Decoder takes the previous output token and the encoder’s hidden state to generate the next token. It has its own embedding layer and GRU, and a linear layer `fc_out` that maps the GRU hidden state to a probability distribution over the target vocabulary. The input is unsqueezed to add a sequence dimension because the decoder processes one time step at a time.

The Seq2Seq wrapper combines the encoder and decoder into a single model. It implements teacher forcing, which randomly chooses between using the true target token (`tgt[t]`) or the model’s prediction (`top1`) as the next input, controlled by `teacher_forcing_ratio`. The model returns all decoder outputs in a tensor of shape `[tgt_len, batch_size, vocab_size]`.

These design choices allow the model to map sequences of tokens between languages efficiently, accelerate training with teacher forcing, and handle variable-length input sequences through embeddings and GRU hidden states.

In [15]:
VOCAB_SIZE = tokenizer.get_vocab_size()

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim)

    def forward(self, src):
        embedded = self.embedding(src) 
        outputs, hidden = self.rnn(embedded)
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden):
        embedded = self.embedding(input.unsqueeze(0))
        output, hidden = self.rnn(embedded, hidden)
        pred = self.fc_out(output.squeeze(0))
        return pred, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[1]
        tgt_len = tgt.shape[0]
        vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(tgt_len, batch_size, vocab_size).to(self.device)

        hidden = self.encoder(src)
        input = tgt[0, :]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[t] if teacher_force else top1

        return outputs

# Model Initialization and Training Loop

This code block sets up the Seq2Seq model, the loss function, the optimizer, and the training loop. The device is set to GPU if available for faster training: 
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```
Model parameters include `VOCAB_SIZE` from the tokenizer, `EMB_DIM = 256` and `HID_DIM = 512`, which define the embedding and hidden state sizes to allow the model to capture more complex patterns. The model is created using 
```python
enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM) 
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)
```

The loss function and optimizer are defined as
```python 
PAD_IDX = tokenizer.token_to_id("<pad>") 
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX) 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
``` 
The `ignore_index=PAD_IDX` ensures that padded tokens do not contribute to the loss, and Adam optimizer provides efficient and adaptive gradient updates.

The training loop iterates over epochs and batches: for each batch, source and target tensors are moved to the device, the model output is computed and reshaped for the cross-entropy loss, gradients are backpropagated, and weights are updated. Gradient clipping (`torch.nn.utils.clip_grad_norm_`) prevents exploding gradients by limiting the norm to 1. The average training loss per epoch is printed to monitor progress. This setup ensures the model learns translations efficiently while maintaining numerical stability and leveraging GPU acceleration.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

VOCAB_SIZE = tokenizer.get_vocab_size()
EMB_DIM = 256
HID_DIM = 512

enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

PAD_IDX = tokenizer.token_to_id("<pad>")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

num_epochs = 50
patience = 2
best_valid_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        tgt = tgt[1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    valid_loss = 0
    with torch.no_grad():
        for src, tgt in valid_loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            tgt = tgt[1:].reshape(-1)
            loss = criterion(output, tgt)
            valid_loss += loss.item()
    valid_loss /= len(valid_loader)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

Using device: cuda


KeyboardInterrupt: 

: 

# Translating a Sentence with the Trained Model

Here I define a function `translate_sentence` to generate a translation for a single Dutch sentence using the trained Seq2Seq model and the tokenizer. The function first sets the model to evaluation mode (`model.eval()`) to disable dropout and gradient computation. The input sentence is tokenized with the tokenizer and truncated to a maximum length (`max_len`) to prevent excessively long sequences. Padding is added with the `<pad>` token to ensure a fixed-length tensor for batch processing. The input tensor is then converted to a PyTorch tensor and moved to the correct device (CPU or GPU).

The encoder processes the source tensor to produce a hidden state. The decoder starts with the `<bos>` token as the first input and generates tokens one at a time, feeding its previous output as the next input. Teacher forcing is not used here; the decoder always uses its own previous prediction. The loop continues until either the `<eos>` token is generated or the maximum length is reached. Each predicted token ID is appended to a list, which is then decoded back into a string using `tokenizer.decode`.

Finally, a random Dutch sentence from `src_texts` is selected for testing, and the predicted English translation is printed. This function demonstrates how the trained model can perform inference on unseen sentences while maintaining the correct sequence mapping and handling padding, start-of-sentence, and end-of-sentence tokens.

In [ ]:
def translate_sentence(model, sentence, tokenizer, max_len=64, device=device):
    model.eval()

    src_ids = tokenizer.encode(sentence).ids
    src_ids = src_ids[:max_len]
    padding = [tokenizer.token_to_id("<pad>")] * (max_len - len(src_ids))
    src_ids += padding

    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device) 

    with torch.no_grad():
        hidden = model.encoder(src_tensor)
        input = torch.tensor([tokenizer.token_to_id("<bos>")]).to(device)
        result_ids = []

        for _ in range(max_len):
            output, hidden = model.decoder(input, hidden)
            top1 = output.argmax(1)
            if top1.item() == tokenizer.token_to_id("<eos>"):
                break
            result_ids.append(top1.item())
            input = top1

    translated = tokenizer.decode(result_ids)
    return translated

import random
test_sentence = random.choice(src_texts)
print("Dutch:", test_sentence)
print("English (pred):", translate_sentence(model, test_sentence, tokenizer))

NameError: name 'device' is not defined

In [ ]:
'''
example = test_sentence = random.choice(src_texts)
ids = tokenizer.encode(example).ids
print("Token IDs:", ids)
print("Decoded:", tokenizer.decode(ids))
'''

Token IDs: [247, 101, 989, 200, 1394, 12]
Decoded: Dat is altijd zo geweest .


In [ ]:
'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

VOCAB_SIZE = tokenizer.get_vocab_size()
EMB_DIM = 256
HID_DIM = 512

enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

PAD_IDX = tokenizer.token_to_id("<pad>")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

num_epochs = 20
patience = 2
best_valid_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        tgt = tgt[1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    valid_loss = 0
    with torch.no_grad():
        for src, tgt in valid_loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            tgt = tgt[1:].reshape(-1)
            loss = criterion(output, tgt)
            valid_loss += loss.item()
    valid_loss /= len(valid_loader)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break
'''